In [1]:
import os
import re
from typing import List, Dict, Optional
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
# === Setup ===
load_dotenv()

True

In [5]:
client = OpenAI(
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [7]:
# === Utility Functions ===

def llm_call(prompt: str, model: str = "gpt-4") -> str:
    """Basic LLM call wrapper"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

In [9]:
def extract_xml(text: str, tag: str) -> str:
    """Extract content between XML-style tags"""
    pattern = rf"<{tag}>(.*?)</{tag}>"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else ""

In [11]:
def parse_tasks(xml: str) -> List[Dict]:
    """Parses <task> XML blocks into dictionaries"""
    tasks = []
    current_task = {}

    for line in xml.splitlines():
        line = line.strip()
        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks

In [13]:
# === Worker Agent Base Class ===

class WorkerAgent:
    def __init__(self, task_type: str):
        self.task_type = task_type

    def run(self, original_task: str, task_description: str) -> str:
        raise NotImplementedError("Must implement run() method in subclass")

In [15]:
# === Worker Agent Implementations ===

class TelecomAgent(WorkerAgent):
    def run(self, original_task: str, task_description: str) -> str:
        prompt = f"""
You are a Telecom expert.

Main Task: {original_task}
Subtask Type: {self.task_type}
Subtask Description: {task_description}

<response>
- Explain the purpose of this step.
- Describe required concepts and steps.
- Note any blockers like conflicts or variances.
</response>
"""
        raw_output = llm_call(prompt)
        result = extract_xml(raw_output, "response")
        if not result:
            print(f"[WARNING] No <response> tag found. Using raw output:\n{raw_output}")
            result = raw_output
        return result

In [17]:
class SitePlanAgent(WorkerAgent):
    def run(self, original_task: str, task_description: str) -> str:
        prompt = f"""
You are a site planning advisor.

Main Task: {original_task}
Subtask Type: {self.task_type}
Subtask Description: {task_description}

<response>
- What does building a 4G or a 5G site planning involve?
- What approvals or drawings are needed?
- Who reviews them and why could there be delays in building a 4G or a 5G site?
</response>
"""
        raw_output = llm_call(prompt)
        result = extract_xml(raw_output, "response")
        if not result:
            print(f"[WARNING] No <response> tag found. Using raw output:\n{raw_output}")
            result = raw_output
        return result

In [19]:
class GenericAgent(WorkerAgent):
    def run(self, original_task: str, task_description: str) -> str:
        prompt = f"""
You are a permitting assistant.

Main Task: {original_task}
Subtask Type: {self.task_type}
Subtask Description: {task_description}

<response>
- Describe this subtask.
- List how to execute it.
- Include potential challenges.
</response>
"""
        raw_output = llm_call(prompt)
        result = extract_xml(raw_output, "response")
        if not result:
            print(f"[WARNING] No <response> tag found. Using raw output:\n{raw_output}")
            result = raw_output
        return result

In [21]:
# === Orchestrator ===

class Orchestrator:
    def __init__(self, orchestrator_prompt: str):
        self.orchestrator_prompt = orchestrator_prompt

    def get_worker(self, task_type: str) -> WorkerAgent:
        type_lower = task_type.lower()
        if "telecom" in type_lower:
            return TelecomAgent(task_type)
        elif "site" in type_lower:
            return SitePlanAgent(task_type)
        else:
            return GenericAgent(task_type)

    def process(self, task: str) -> Dict:
        # Step 1: Decompose task using orchestrator LLM
        orchestrator_input = self.orchestrator_prompt.format(task=task)
        response = llm_call(orchestrator_input)
        print("\n[Raw Orchestrator Output]\n", response)

        analysis = extract_xml(response, "analysis")
        tasks_xml = extract_xml(response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("\n=== ORCHESTRATOR ===")
        print("Analysis:", analysis)
        print("Parsed Tasks:", tasks)

        # Step 2: Dispatch tasks to worker agents
        results = []
        for task_info in tasks:
            agent = self.get_worker(task_info["type"])
            result = agent.run(task, task_info["description"])
            print(f"\n=== {task_info['type'].upper()} RESULT ===\n{result}")
            results.append({
                "type": task_info["type"],
                "description": task_info["description"],
                "result": result
            })

        return {
            "analysis": analysis,
            "worker_results": results
        }

In [23]:
# === Prompt Template for Orchestrator ===

orchestrator_prompt = """
You are a Telecom expert. Your job is to analyze a high-level construction request of a 4G or a 5G Site and break it into permitting-related subtasks.

<analysis>
Provide an overall explanation of what steps are involved.
</analysis>

<tasks>
Provide 2–4 <task> entries with a <type> and <description>. Use this format:
<task>
  <type>telecom</type>
  <description>Verify all necessary steps for permits etc are understood.</description>
</task>
</tasks>

Task: {task}
"""

In [25]:
# === Main Runner ===

if __name__ == "__main__":
    user_prompt = "What Steps are involved in building a Telecom 4G or a 5G Site?"

    orchestrator = Orchestrator(orchestrator_prompt)
    results = orchestrator.process(user_prompt)

    print("\n\n=== FINAL REPORT ===")
    print("Analysis:\n", results["analysis"])
    for r in results["worker_results"]:
        print(f"\n--- {r['type'].upper()} ---")
        print("Description:", r["description"])
        print("Result:\n", r["result"])



[Raw Orchestrator Output]
 <analysis>
Building a Telecom 4G or 5G site involves several steps, many of which are related to obtaining the necessary permits. The process begins with site identification and acquisition, where the best location for the site is determined and secured. This is followed by the design and planning phase, where the specifics of the site, such as the type of tower and equipment to be used, are determined. 

The next step is the permitting phase, where all necessary permits are obtained. This can include zoning permits, building permits, and environmental permits, among others. Each of these permits has its own requirements and processes, which must be understood and followed. 

Once all permits are obtained, construction can begin. This involves the physical building of the site, including the installation of the tower and equipment. After construction is completed, the site must be tested and commissioned before it can be operational. 

Throughout this proces